[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/40.%20The%20Port%20Capacity%20%26%20Expansion%20Timing%20Problem/P40-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 40. The Port Capacity and Expansion Timing Problem
## Tier 2: A Randomized Variable Neighborhood Descent Heuristic

### Goal and inputs
This notebook uses Variable Neighborhood Descent (VND) to improve a port capacity expansion plan. A plan specifies the expansion year, capacity increase, technology, and whether construction is phased. The model evaluates each plan using investment cost, discounted operational benefits, and discounted congestion savings. The search starts from a specified plan and accepts strictly improving changes.

The supplied example uses a 2020 reference year. An expansion-year decision of 6 therefore corresponds to calendar year 2026. Capacity and demand are measured in million TEU; monetary quantities in the calculations are in millions of dollars.

### What VND does
A neighborhood is a set of plans that can be reached by a particular type of change. This implementation has four neighborhoods: adjust timing, change capacity, change technology, and toggle phasing. At each step it samples one neighboring plan from the current neighborhood. If that plan has a better objective value, the algorithm accepts it and restarts with the first neighborhood. Otherwise, it tries the next neighborhood. The procedure stops after a complete pass with no accepted move or when the accepted-move limit is reached.

This is a randomized, single-neighbor VND heuristic: it samples one move per neighborhood visit rather than exhaustively evaluating every plan in a neighborhood. Consequently, stopping means that the sampled pass found no improvement; it does not certify that the solution is a local optimum over all possible neighbors. The random seed makes the sequence of sampled moves reproducible in a run with the same software environment.

### Model assumptions and scope
- Capacity choices are 15%, 40%, and 85%; expansion timing is represented by years 1 through 10 from the 2020 reference year.
- Operational benefit is a simplified annual benefit based on added capacity, revenue per TEU, and a technology efficiency factor. It is not capped by a separate demand-allocation model.
- Congestion savings use the utilization calculated at the expansion year and hold the resulting annual saving constant through the remaining modeled years. The congestion penalty is treated as dollars per day and converted to million dollars per year before it is combined with the other financial quantities.
- Phasing affects investment cost through a premium. Phase duration is recorded as a decision attribute but does not change the benefit schedule in this simplified model.
- The model does not represent taxes, financing structure, construction disruption, uncertainty scenarios, or detailed capacity ramp-up. Its objective is a planning heuristic, not a complete project appraisal.

### Mathematical model
Let a plan be represented by the decision tuple
$$x=(e,q,z,p,d),$$
where $e$ is the expansion year, $q$ is the fractional capacity increase, $z$ is the technology, $p$ indicates whether the project is phased, and $d$ is the phase duration. The plan's investment cost is $I(x)$. The search minimizes the negative of net present value, equivalently minimizing
$$f(x)=I(x)-B_{\mathrm{op}}(x)-S_{\mathrm{cong}}(x).$$
Here, $B_{\mathrm{op}}$ is the present value of operational benefits, and $S_{\mathrm{cong}}$ is the present value of congestion savings. Thus, a lower objective value means a higher net present value.

Write $C$ for current capacity, $r$ for revenue per TEU, $\eta_z$ for the technology efficiency factor, $i$ for the discount rate, and $H$ for the planning horizon. The added capacity is $\Delta C=Cq$. The operational benefit is
$$B_{\mathrm{op}}(x)=\sum_{t=e}^{H-1}\frac{\Delta C\,r\,\eta_z}{(1+i)^t}.$$
Because $\Delta C$ is in million TEU and $r$ is in dollars per TEU, the product is in million dollars per year.

Let $D_0$ be current demand, $g$ the annual demand growth rate, and $u$ utilization. At expansion year $e$, utilization without expansion and with expansion is
$$u_0=\min\!\left(\frac{D_0(1+g)^e}{C},1\right),\qquad
u_1=\min\!\left(\frac{D_0(1+g)^e}{C(1+q)},1\right).$$
The annual congestion penalty, expressed in million dollars, is
$$P(u)=\begin{cases}
0, & u\leq 0.8,\\
\dfrac{1000\times365}{10^6}\exp\!\bigl(2(u-0.8)\bigr), & u>0.8.
\end{cases}$$
The factor 1000 represents dollars per day; multiplying by 365 gives dollars per year, and dividing by $10^6$ converts the result to million dollars per year. With annual saving $s=P(u_0)-P(u_1)$, the model uses
$$S_{\mathrm{cong}}(x)=\sum_{t=e}^{H-1}\frac{s}{(1+i)^t}.$$

The VND search maintains a current plan $x$ and its objective $f(x)$. For neighborhood $N_k$, it samples a neighbor $y\in N_k(x)$. If $f(y)<f(x)$, it sets $x\leftarrow y$ and restarts at $k=1$; otherwise it advances to $k+1$. A full pass through all four neighborhoods without an accepted move terminates the search. The code's `evaluate_solution` computes $f$, the neighborhood methods sample the changes, and `solve` implements this acceptance and restart rule.

If at most $M$ moves are accepted, there are four neighborhoods, and evaluating a plan sums over at most $H$ years, the work is on the order of $O((M+1)4H)$ objective calculations at most. This bound describes the implemented sampling procedure; it does not imply exhaustive search or global optimality.

### Example plan and interpretation
The starting plan expands in 2026, increases capacity by 40%, uses semi-automated technology, and is not phased. Its investment cost follows directly from the model's 40% base cost of $450 million and the semi-automated multiplier of 1.2, giving $540 million. The notebook computes the objective and resulting plan when its code is run; no particular search outcome is guaranteed because the neighborhoods are sampled.

The method is useful for illustrating how structured local changes can improve a capacity plan without enumerating every combination. Its result is a heuristic solution, not a proof of optimality. Solution quality depends on the objective model, neighborhood design, starting plan, and sampled moves.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns

# Fix both random-number generators so sampled neighborhood moves can be reproduced.
np.random.seed(42)
random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

print("Libraries imported successfully for the VND heuristic.")

In [ ]:
@dataclass
class ExpansionSolution:
    """Represent one port capacity expansion plan."""

    expansion_year: int
    capacity_increase: float
    technology: str
    phased: bool
    phase_duration: int = 3

    def get_total_cost(self) -> float:
        """Return investment cost in millions of dollars."""
        base_costs = {0.15: 120, 0.40: 450, 0.85: 750}
        technology_multipliers = {
            "manual": 1.0,
            "semi_automated": 1.2,
            "fully_automated": 1.5,
        }

        base_cost = base_costs.get(self.capacity_increase, 450)
        technology_multiplier = technology_multipliers.get(self.technology, 1.0)
        phasing_multiplier = 1.15 if self.phased else 1.0

        return base_cost * technology_multiplier * phasing_multiplier


@dataclass
class PortParameters:
    """Store the port and financial inputs used to evaluate plans."""

    current_capacity: float = 3.2
    current_demand: float = 2.8
    revenue_per_teu: float = 180
    annual_growth_rate: float = 0.07
    discount_rate: float = 0.08
    planning_horizon: int = 15


class VariableNeighborhoodDescent:
    """Apply randomized single-neighbor Variable Neighborhood Descent."""

    def __init__(self, port_params: PortParameters):
        self.port_params = port_params
        self.neighborhood_functions = [
            self._neighborhood_timing,
            self._neighborhood_capacity,
            self._neighborhood_technology,
            self._neighborhood_phasing,
        ]
        self.iteration_history = []

    def _neighborhood_timing(self, solution: ExpansionSolution) -> ExpansionSolution:
        """Sample a plan with its expansion year shifted by one year."""
        neighbor = deepcopy(solution)
        year_change = random.choice([-1, 1])
        neighbor.expansion_year = max(1, min(10, neighbor.expansion_year + year_change))
        return neighbor

    def _neighborhood_capacity(self, solution: ExpansionSolution) -> ExpansionSolution:
        """Sample an adjacent capacity choice in the cyclic option list."""
        neighbor = deepcopy(solution)
        capacity_options = [0.15, 0.40, 0.85]
        current_index = capacity_options.index(neighbor.capacity_increase)
        new_index = (current_index + random.choice([-1, 1])) % len(capacity_options)
        neighbor.capacity_increase = capacity_options[new_index]
        return neighbor

    def _neighborhood_technology(self, solution: ExpansionSolution) -> ExpansionSolution:
        """Sample a technology different from the plan's current technology."""
        neighbor = deepcopy(solution)
        technology_options = ["manual", "semi_automated", "fully_automated"]
        neighbor.technology = random.choice(
            [technology for technology in technology_options if technology != neighbor.technology]
        )
        return neighbor

    def _neighborhood_phasing(self, solution: ExpansionSolution) -> ExpansionSolution:
        """Toggle phasing and sample a duration when the plan is phased."""
        neighbor = deepcopy(solution)
        neighbor.phased = not neighbor.phased

        if neighbor.phased:
            neighbor.phase_duration = random.choice([2, 3, 4])

        return neighbor

    def evaluate_solution(self, solution: ExpansionSolution) -> float:
        """Return the negative net present value in millions of dollars."""
        investment_cost = solution.get_total_cost()
        operational_benefit = self._calculate_operational_benefit(solution)
        congestion_savings = self._calculate_congestion_savings(solution)
        net_present_value = operational_benefit + congestion_savings - investment_cost

        # Minimizing negative NPV is equivalent to maximizing NPV.
        return -net_present_value

    def _calculate_operational_benefit(self, solution: ExpansionSolution) -> float:
        """Discount the simplified annual benefit from added capacity."""
        years_to_expansion = solution.expansion_year
        new_capacity = self.port_params.current_capacity * (
            1 + solution.capacity_increase
        )
        additional_capacity = new_capacity - self.port_params.current_capacity

        technology_efficiency = {
            "manual": 1.0,
            "semi_automated": 1.15,
            "fully_automated": 1.30,
        }
        efficiency_gain = technology_efficiency.get(solution.technology, 1.0)
        annual_revenue = (
            additional_capacity
            * self.port_params.revenue_per_teu
            * efficiency_gain
        )

        present_value = 0.0
        for year in range(years_to_expansion, self.port_params.planning_horizon):
            present_value += annual_revenue / (
                (1 + self.port_params.discount_rate) ** year
            )

        return present_value

    def _calculate_congestion_savings(self, solution: ExpansionSolution) -> float:
        """Discount annual congestion savings, in millions of dollars."""
        years_to_expansion = solution.expansion_year
        new_capacity = self.port_params.current_capacity * (
            1 + solution.capacity_increase
        )

        future_demand = self.port_params.current_demand * (
            (1 + self.port_params.annual_growth_rate) ** years_to_expansion
        )
        utilization_without_expansion = min(
            future_demand / self.port_params.current_capacity,
            1.0,
        )
        utilization_with_expansion = min(future_demand / new_capacity, 1.0)

        def congestion_penalty(utilization: float) -> float:
            """Return the annual congestion penalty in millions of dollars."""
            if utilization <= 0.8:
                return 0.0

            dollars_per_day = 1000 * np.exp(2 * (utilization - 0.8))
            return dollars_per_day * 365 / 1_000_000

        annual_savings = congestion_penalty(
            utilization_without_expansion
        ) - congestion_penalty(utilization_with_expansion)

        present_value = 0.0
        for year in range(years_to_expansion, self.port_params.planning_horizon):
            present_value += annual_savings / (
                (1 + self.port_params.discount_rate) ** year
            )

        return present_value

    def solve(
        self,
        initial_solution: ExpansionSolution,
        max_iterations: int = 100,
    ) -> Tuple[ExpansionSolution, float, List]:
        """Search sampled neighborhoods and accept strictly improving plans."""
        current_solution = deepcopy(initial_solution)
        current_cost = self.evaluate_solution(current_solution)

        self.iteration_history = [
            {
                "iteration": 0,
                "solution": deepcopy(current_solution),
                "cost": current_cost,
                "neighborhood": -1,
                "improvement": 0.0,
            }
        ]

        iteration = 0
        improved = True

        while improved and iteration < max_iterations:
            improved = False

            for neighborhood_index, neighborhood_function in enumerate(
                self.neighborhood_functions
            ):
                neighbor_solution = neighborhood_function(current_solution)
                neighbor_cost = self.evaluate_solution(neighbor_solution)

                if neighbor_cost < current_cost:
                    improvement = current_cost - neighbor_cost
                    current_solution = neighbor_solution
                    current_cost = neighbor_cost
                    improved = True
                    iteration += 1
                    self.iteration_history.append(
                        {
                            "iteration": iteration,
                            "solution": deepcopy(current_solution),
                            "cost": current_cost,
                            "neighborhood": neighborhood_index + 1,
                            "improvement": improvement,
                        }
                    )

                    # Restarting at the first neighborhood is the VND change rule.
                    break

        return current_solution, -current_cost, self.iteration_history


print("VND algorithm classes defined successfully.")

In [ ]:
# Define the port inputs and the starting plan.
port_params = PortParameters()
reference_year = 2020

initial_solution = ExpansionSolution(
    expansion_year=6,
    capacity_increase=0.40,
    technology="semi_automated",
    phased=False,
    phase_duration=3,
)

print("INITIAL SOLUTION CONFIGURATION:")
print(f"Expansion year: {reference_year + initial_solution.expansion_year}")
print(f"Capacity increase: {initial_solution.capacity_increase:.0%}")
print(f"Technology: {initial_solution.technology}")
print(f"Phased: {initial_solution.phased}")
print(f"Investment cost: ${initial_solution.get_total_cost():.1f} million")

vnd_solver = VariableNeighborhoodDescent(port_params)
initial_objective = vnd_solver.evaluate_solution(initial_solution)
initial_npv = -initial_objective

print(f"\nInitial net present value: ${initial_npv:.3f} million")
print(f"Initial minimization objective (negative NPV): {initial_objective:.3f}")

In [ ]:
print("\n" + "=" * 60)
print("RUNNING VARIABLE NEIGHBORHOOD DESCENT")
print("=" * 60)

final_solution, final_npv, iteration_history = vnd_solver.solve(
    initial_solution,
    max_iterations=50,
)

number_of_moves = len(iteration_history) - 1
npv_improvement = final_npv - initial_npv
improvement_percentage = (
    100 * npv_improvement / abs(initial_npv) if initial_npv != 0 else float("nan")
)

print(f"\nAccepted improving moves: {number_of_moves}")
print(f"Final net present value: ${final_npv:.3f} million")
print(
    f"NPV improvement: ${npv_improvement:.3f} million "
    f"({improvement_percentage:.2f}%)"
)

print("\n" + "=" * 60)
print("FINAL SEARCH PLAN")
print("=" * 60)
print(f"Expansion year: {reference_year + final_solution.expansion_year}")
print(f"Capacity increase: {final_solution.capacity_increase:.0%}")
print(f"Technology: {final_solution.technology}")
print(f"Phased: {final_solution.phased}")
if final_solution.phased:
    print(f"Phase duration: {final_solution.phase_duration} years")
print(f"Investment cost: ${final_solution.get_total_cost():.1f} million")
print(f"Net present value: ${final_npv:.3f} million")

# Count accepted moves by neighborhood to show which sampled changes were useful.
neighborhood_names = ["Timing", "Capacity", "Technology", "Phasing"]
neighborhood_improvements = {name: 0.0 for name in neighborhood_names}
neighborhood_counts = {name: 0 for name in neighborhood_names}

for record in iteration_history[1:]:
    if record["improvement"] > 0:
        neighborhood_name = neighborhood_names[record["neighborhood"] - 1]
        neighborhood_improvements[neighborhood_name] += record["improvement"]
        neighborhood_counts[neighborhood_name] += 1

print("\nACCEPTED IMPROVEMENTS BY NEIGHBORHOOD")
for name in neighborhood_names:
    count = neighborhood_counts[name]
    if count:
        average_improvement = neighborhood_improvements[name] / count
        print(
            f"{name}: {count} accepted moves, "
            f"average objective reduction ${average_improvement:.3f} million"
        )
    else:
        print(f"{name}: no accepted moves")

In [ ]:
print("\n" + "=" * 80)
print("DETAILED ITERATION ANALYSIS")
print("=" * 80)

analysis_data = []
for record in iteration_history:
    solution = record["solution"]
    if record["iteration"] == 0:
        neighborhood_name = "Initial"
    else:
        neighborhood_name = neighborhood_names[record["neighborhood"] - 1]

    analysis_data.append(
        {
            "Iteration": record["iteration"],
            "Neighborhood": neighborhood_name,
            "Expansion year": reference_year + solution.expansion_year,
            "Capacity increase": f"{solution.capacity_increase:.0%}",
            "Technology": solution.technology,
            "Phased": solution.phased,
            "Investment cost ($M)": solution.get_total_cost(),
            "NPV ($M)": -record["cost"],
            "Objective reduction ($M)": record["improvement"],
        }
    )

analysis_df = pd.DataFrame(analysis_data)
print(analysis_df.head(10).to_string(index=False))
if len(analysis_df) > 10:
    print(f"\n{len(analysis_df) - 10} additional rows are not shown.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Variable Neighborhood Descent Search Results",
    fontsize=16,
    fontweight="bold",
)

iterations = [record["iteration"] for record in iteration_history]
npvs = [-record["cost"] for record in iteration_history]

# The first panel tracks the objective's NPV interpretation across accepted moves.
axes[0, 0].plot(iterations, npvs, "b-o", linewidth=2, markersize=6)
axes[0, 0].set_title("Net Present Value by Accepted Move")
axes[0, 0].set_xlabel("Accepted move")
axes[0, 0].set_ylabel("Net present value (million dollars)")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(
    y=initial_npv,
    color="red",
    linestyle="--",
    alpha=0.7,
    label="Starting plan",
)
axes[0, 0].legend()

# Each bar reports the accumulated objective reduction from accepted moves.
contributing_neighborhoods = [
    name for name in neighborhood_names if neighborhood_counts[name] > 0
]
if contributing_neighborhoods:
    contributions = [
        neighborhood_improvements[name] for name in contributing_neighborhoods
    ]
    bars = axes[0, 1].bar(contributing_neighborhoods, contributions)
    axes[0, 1].set_title("Accepted Objective Reduction by Neighborhood")
    axes[0, 1].set_ylabel("Objective reduction (million dollars)")
    axes[0, 1].tick_params(axis="x", rotation=35)
    for bar, contribution in zip(bars, contributions):
        axes[0, 1].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"${contribution:.2f}M",
            ha="center",
            va="bottom",
        )
else:
    axes[0, 1].text(
        0.5,
        0.5,
        "No improving moves were accepted",
        ha="center",
        va="center",
        transform=axes[0, 1].transAxes,
    )
    axes[0, 1].set_title("Accepted Objective Reduction by Neighborhood")

# Capacity is a discrete decision, so its available levels are shown explicitly.
capacities = [
    100 * record["solution"].capacity_increase for record in iteration_history
]
axes[1, 0].plot(iterations, capacities, "o-", linewidth=2, markersize=8)
axes[1, 0].set_title("Capacity Increase in the Search Plan")
axes[1, 0].set_xlabel("Accepted move")
axes[1, 0].set_ylabel("Capacity increase (percent)")
axes[1, 0].set_yticks([15, 40, 85])
axes[1, 0].grid(True, alpha=0.3)

expansion_years = [
    reference_year + record["solution"].expansion_year
    for record in iteration_history
]
axes[1, 1].plot(iterations, expansion_years, "g-o", linewidth=2, markersize=6)
axes[1, 1].set_title("Expansion Year in the Search Plan")
axes[1, 1].set_xlabel("Accepted move")
axes[1, 1].set_ylabel("Calendar year")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(
    y=reference_year + initial_solution.expansion_year,
    color="red",
    linestyle="--",
    alpha=0.7,
    label="Starting year",
)
axes[1, 1].legend()

plt.tight_layout()
plt.show()
print("Search-result visualizations are complete.")

In [ ]:
def analyze_solution_detailed(
    solution: ExpansionSolution,
    name: str,
    solver: VariableNeighborhoodDescent,
    reference_year: int,
) -> dict:
    """Report objective components for a plan using the supplied model."""
    investment_cost = solution.get_total_cost()
    net_present_value = -solver.evaluate_solution(solution)
    operational_benefit = solver._calculate_operational_benefit(solution)
    congestion_savings = solver._calculate_congestion_savings(solution)

    print(f"\n{name} SOLUTION ANALYSIS:")
    print(f"  Expansion year: {reference_year + solution.expansion_year}")
    print(f"  Capacity increase: {solution.capacity_increase:.0%}")
    print(f"  Technology: {solution.technology}")
    print(f"  Phased: {solution.phased}")
    if solution.phased:
        print(f"  Phase duration: {solution.phase_duration} years")
    print(f"  Investment cost: ${investment_cost:.3f} million")
    print(f"  Operational benefit: ${operational_benefit:.3f} million")
    print(f"  Congestion savings: ${congestion_savings:.3f} million")
    print(f"  Net present value: ${net_present_value:.3f} million")

    return {
        "cost": investment_cost,
        "npv": net_present_value,
        "operational_benefit": operational_benefit,
        "congestion_savings": congestion_savings,
    }


print("\n" + "=" * 80)
print("STARTING-PLAN AND SEARCH-PLAN COMPARISON")
print("=" * 80)

initial_analysis = analyze_solution_detailed(
    initial_solution,
    "STARTING",
    vnd_solver,
    reference_year,
)
final_analysis = analyze_solution_detailed(
    final_solution,
    "SEARCH",
    vnd_solver,
    reference_year,
)

npv_improvement = final_analysis["npv"] - initial_analysis["npv"]
cost_change = final_analysis["cost"] - initial_analysis["cost"]
operational_improvement = (
    final_analysis["operational_benefit"]
    - initial_analysis["operational_benefit"]
)
congestion_improvement = (
    final_analysis["congestion_savings"]
    - initial_analysis["congestion_savings"]
)

print("\n" + "=" * 60)
print("CHANGE IN MODEL RESULTS")
print("=" * 60)
print(f"NPV change: ${npv_improvement:.3f} million")
print(f"Investment-cost change: ${cost_change:.3f} million")
print(f"Operational-benefit change: ${operational_improvement:.3f} million")
print(f"Congestion-savings change: ${congestion_improvement:.3f} million")
if initial_analysis["npv"] != 0:
    relative_improvement = (
        100 * npv_improvement / abs(initial_analysis["npv"])
    )
    print(f"NPV change relative to the starting plan: {relative_improvement:.2f}%")

print("\nINTERPRETATION")
if final_solution.expansion_year > initial_solution.expansion_year:
    print("The sampled search plan schedules expansion later than the starting plan.")
elif final_solution.expansion_year < initial_solution.expansion_year:
    print("The sampled search plan schedules expansion earlier than the starting plan.")

if final_solution.capacity_increase > initial_solution.capacity_increase:
    print("The sampled search plan chooses a larger capacity increase.")
elif final_solution.capacity_increase < initial_solution.capacity_increase:
    print("The sampled search plan chooses a smaller capacity increase.")

if final_solution.technology != initial_solution.technology:
    print(f"The sampled search plan selects {final_solution.technology} technology.")

if final_solution.phased != initial_solution.phased:
    phasing_description = "phased" if final_solution.phased else "single-phase"
    print(f"The sampled search plan uses a {phasing_description} implementation.")

print(
    "The reported plan is the result of the sampled VND search and is not "
    "a certificate of global optimality."
)
print("\n" + "=" * 80)